# SD 1.5 (.safetensors) → Qualcomm QNN `qnn2.39_min` — Colab

Bir **safetensors indirme linki** girin → dönüştürün → **Hugging Face reponuza** yükleyin.
Çıktı: `<isim>_qnn2.39_min.zip` (Ruya / Local Dream ile içe aktarılır).

### Önce oku (önemli):
1. **Runtime → Change runtime type → High-RAM** seçin. Ücretsiz katman (12 GB) 512px'te OOM olabilir; **Colab Pro / High-RAM (25–51 GB)** önerilir.
2. **QNN/QAIRT SDK otomatik indirilir** — `matrixportalx/qairt-sdk` **v2.39.0.250926** release'inden. Public olduğu için token gerekmez.
3. QAIRT araçları **Python 3.10** ister (Colab 3.12); 4. adım izole bir 3.10 ortamı kurar (otomatik).
4. HF token'ınızı Colab **Secrets** (🔑 sol menü) içine `HF_TOKEN` adıyla ekleyin (write izinli).
5. civitai linki token istiyorsa Secrets'a `CIVITAI_TOKEN` ekleyin.

Hücreleri **sırayla** çalıştırın.

## 1) Ayarlar (buradan doldurun)

In [ ]:
#@title Dönüşüm ayarları { display-mode: "form" }
SAFETENSORS_URL = ""  #@param {type:"string"}
MODEL_NAME = "MyModel"  #@param {type:"string"}
TIER = "min"  #@param ["min", "mid", "high"]
RESOLUTIONS = "512x512"  #@param ["512x512", "512x512,512x768,768x512"] {allow-input: true}

#@markdown **Hugging Face yükleme modu**
#@markdown - *Ayrı repo*: her model kendi reposuna → `<kullanıcı>/MODEL_NAME`
#@markdown - *Koleksiyon*: hepsi tek repoda, her model alt klasörde → `<kullanıcı>/COLLECTION/MODEL_NAME/`
UPLOAD_MODE = "Ayri repo (model adiyla)"  #@param ["Ayri repo (model adiyla)", "Koleksiyon (tek repo)"]
COLLECTION_REPO = "sd_qnn"  #@param {type:"string"}
HF_REPO = ""  #@param {type:"string"}
HF_PRIVATE = False  #@param {type:"boolean"}
#@markdown (`HF_REPO` doldurulursa iki modu da geçersiz kılar; doğrudan o repoya yükler.)

#@markdown **QAIRT SDK release** (varsayılan sizin public release'iniz)
QAIRT_REPO = "matrixportalx/qairt-sdk"  #@param {type:"string"}
QAIRT_TAG = "v2.39.0.250926"  #@param {type:"string"}
QNN_VERSION = "2.39"  #@param {type:"string"}

import os
os.environ["QNN_VERSION"] = QNN_VERSION
assert SAFETENSORS_URL, "SAFETENSORS_URL boş olamaz"
print("Ayarlar tamam:", MODEL_NAME, TIER, RESOLUTIONS, "| qnn", QNN_VERSION)

## 2) Depoyu çek + Python bağımlılıkları + MNN

In [ ]:
%cd /content
BR = "claude/qnn-model-conversion-snapdragon7-rsk8og"
URL = "https://github.com/matrixportalx/Sd-1.5-Converting-to-Qualcomm-QNN-Model.git"
# Klasor varsa en guncel kodu cek; yoksa klonla (re-run'da guncel requirements alinir)
!if [ -d sd-qnn ]; then cd sd-qnn && git fetch -q origin $BR && git reset -q --hard origin/$BR && cd ..; else git clone -q --branch $BR $URL sd-qnn; fi
%cd /content/sd-qnn
!pip -q install -r requirements.txt
# MNN converter (text_encoder + vae -> .mnn) + onnx export/sadelestirme araclari
!pip -q install MNN onnxscript onnxslim
import os
os.environ["MNNCONVERT"] = "mnnconvert"
print("OK")

## 3) (İsteğe bağlı) Swap ekle — düşük RAM'de OOM'u azaltır

In [ ]:
#@title 16 GB swap oluştur (ücretsiz katmanda önerilir)
!fallocate -l 16G /content/swapfile 2>/dev/null || dd if=/dev/zero of=/content/swapfile bs=1M count=16384
!chmod 600 /content/swapfile && mkswap /content/swapfile && swapon /content/swapfile
!free -h

## 4) QAIRT 2.39 SDK'yı indir + Python 3.10 ortamını kur (otomatik)

SDK release'ten iner; QAIRT araçları için izole bir Python 3.10 venv + libc++ kurulur. Release private ise Colab Secrets'a `GH_TOKEN` ekleyin (public ise gerekmez).

In [ ]:
import os
try:
    from google.colab import userdata
    for k in ("GH_TOKEN", "GITHUB_TOKEN"):
        try:
            v = userdata.get(k)
            if v: os.environ["GH_TOKEN"] = v
        except Exception:
            pass
except Exception:
    pass

!python scripts/setup_qnn_sdk.py --repo "$QAIRT_REPO" --tag "$QAIRT_TAG" --dest /content/qairt | tee /content/sdk_setup.log

root = None
for line in open("/content/sdk_setup.log"):
    if line.startswith("QNN_SDK_ROOT="):
        root = line.strip().split("=", 1)[1]
assert root, "QNN_SDK_ROOT bulunamadı — 4. adım loguna bakın."
os.environ["QNN_SDK_ROOT"] = root
print("QNN_SDK_ROOT =", root)

# QAIRT python konvertorleri Python 3.10 + libc++ ister -> izole venv kur
!chmod +x scripts/*.sh
!bash scripts/setup_qnn_python.sh
if os.path.exists("/content/qnn_py.path"):
    os.environ["QNN_PYTHON"] = open("/content/qnn_py.path").read().strip()
    print("QNN_PYTHON =", os.environ["QNN_PYTHON"])

## 5) Modeli indir (civitai / HF / düz link)

In [ ]:
import os
try:
    from google.colab import userdata
    for k in ("CIVITAI_TOKEN", "HF_TOKEN"):
        try:
            v = userdata.get(k)
            if v: os.environ[k] = v
        except Exception:
            pass
except Exception:
    pass

!python scripts/fetch_model.py --url "$SAFETENSORS_URL" --output work/input.safetensors

## 6) Dönüştür (uçtan uca)

**Uyarı:** Bu adım en uzunudur — çözünürlük başına **saatler** sürebilir. Colab oturumu kopmasın diye sekmeyi açık tutun.

In [ ]:
!chmod +x convert_all.sh scripts/*.sh
!./convert_all.sh work/input.safetensors "$MODEL_NAME" "$TIER" "$RESOLUTIONS"
import glob
print("Üretilen:", glob.glob("dist/*.zip"))

## 7) Hugging Face'e yükle

`HF_REPO` boşsa, **model isminden otomatik repo** oluşturulur: `<kullanıcı_adın>/MODEL_NAME` — ve model oraya yüklenir (küçük bir model kartıyla). Belirli bir repoya yüklemek istersen `HF_REPO` alanını doldur. Gerekli: Colab Secrets'ta write izinli `HF_TOKEN`.

In [ ]:
import glob, os
zips = sorted(glob.glob("dist/*.zip"))
assert zips, "dist/ içinde zip yok — 6. adım başarısız olmuş olabilir."
zip_path = zips[-1]

# HF_TOKEN'i Secrets'tan al (yazma izinli olmalı)
try:
    from google.colab import userdata
    t = userdata.get("HF_TOKEN")
    if t:
        os.environ["HF_TOKEN"] = t
except Exception:
    pass

priv = "--private" if HF_PRIVATE else ""
if HF_REPO.strip():
    # Doğrudan verilen repoya yükle
    cmd = f'python scripts/upload_hf.py --repo "{HF_REPO}" --name "{MODEL_NAME}" --file "{zip_path}" {priv}'
elif UPLOAD_MODE.startswith("Koleksiyon"):
    # MOD B: tek koleksiyon reposu, her model alt klasörde
    cmd = f'python scripts/upload_hf.py --collection "{COLLECTION_REPO}" --name "{MODEL_NAME}" --file "{zip_path}" {priv}'
else:
    # MOD A: model adından ayrı repo
    cmd = f'python scripts/upload_hf.py --name "{MODEL_NAME}" --file "{zip_path}" {priv}'
print(">", cmd)
os.system(cmd)
print("ZIP:", zip_path)

## 8) (İsteğe bağlı) ZIP'i doğrudan bilgisayara indir

In [ ]:
from google.colab import files
import glob
zips = sorted(glob.glob("dist/*.zip"))
if zips:
    files.download(zips[-1])